# A more thorough ML evaluation 

This is a notebook, I'm using to test/recreate exisiting evaluation scripts from the ClimSim code to get to work with my pipeline.

## Setup and collecting data/model

In [ ]:
import torch
import models
import os
from omegaconf import OmegaConf
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
import data_preparation
import logging 




In [2]:
path_to_model_folder = '/home/users/bradlesc/projects/ClimSim/logs/p2.1.3/6/testing/quick_test/2025-11-25-16-20-20/yus_mlp_group_0' 
model_checkpoint_path = os.path.join(path_to_model_folder, 'yus_mlp_None_2025-11-25-16-20-20.ckpt')
model_params_path = os.path.join(path_to_model_folder, 'run_config.yaml')
 

In [3]:
initialize(version_base=None, config_path="../config")
cfg = compose(config_name="train_general.yaml")
print(cfg.dataset)


{'dataset_name': 'climsim_from_raw', 'input_dim': 124, 'output_dim': 128, 'base_folder_path': '/gws/nopw/j04/iecdt/bstanleyclamp/ClimSim_lowres/train/', 'path_to_grid_info': '/home/users/bradlesc/projects/ClimSim/grid_info/ClimSim_low-res_grid-info.nc', 'output_scale_file_path': '/home/users/bradlesc/projects/ClimSim/preprocessing/normalizations/outputs/output_scale.nc', 'target_years': ['0001', '0002', '0003'], 'target_months': ['02'], 'v1_inputs': ['state_t', 'state_q0001', 'state_ps', 'pbuf_SOLIN', 'pbuf_LHFLX', 'pbuf_SHFLX'], 'v1_targets': ['ptend_t', 'ptend_q0001', 'cam_out_NETSW', 'cam_out_FLWDS', 'cam_out_PRECSC', 'cam_out_PRECC', 'cam_out_SOLS', 'cam_out_SOLL', 'cam_out_SOLSD', 'cam_out_SOLLD'], 'dataset_testing_sample_rates': {'unit_test': 1, 'quick': 1000, 'reduced': 100, 'full': 7}, 'group_method': 'group_by_months', 'group_by_months': {'num_groups': 3, 'target_group': False, 'groups': [['12', '01', '02'], ['03', '04', '05'], ['06', '07', '08']], 'test_group': ['09', '10', '

In [4]:
model_params = OmegaConf.load(model_params_path)

model = models.load_model_from_checkpoint(model_checkpoint_path, model_name='yus_mlp', model_params=model_params, data_params=cfg.dataset)

In [6]:
trainset = data_preparation.get_dataset(cfg.dataset, 'train', 'quick', 'yus_mlp')
testset = data_preparation.get_dataset(cfg.dataset, 'test', 'quick', 'yus_mlp', trainset.normalisation_stats)

In [9]:
testloader = torch.utils.data.DataLoader(testset, batch_size=len(testset), shuffle=False)

preds = []
targets = []

model.eval()
with torch.no_grad():
    for batch in testloader:
        inputs, target = batch
        output = model(inputs)
        preds.append(output)
        targets.append(target)
preds = torch.cat(preds, dim=0)
targets = torch.cat(targets, dim=0)

print("Predictions shape:", preds.shape)
print("Targets shape:", targets.shape)


Predictions shape: torch.Size([3840, 128])
Targets shape: torch.Size([3840, 128])
